In [3]:
# ===================== 环境配置 =====================
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from jax import flatten_util
import matplotlib.pyplot as plt
from tqdm import tqdm
from jax import vmap,jit,grad
from collections import Counter  # 你要的 count 工具
from NES_VMC import create_machine,hi_ext,hi,SingleStateAnsatz,\
    NESTotalAnsatz,ha,E_fcis, \
    nes_vmc_gradient,make_metropolis_hastings_step,mcmc_sampler_multichain,init_sampler_state,\
    compute_local_energy_matrix_batch,NES_loss_energy,compute_nes_qgt

K = 2
tensor_edges = [(0,1),(2,3),(4,5),(6,7)]
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4,n_states=3,hidden_dim=12,rngs=nnx.Rngs(13))
hi_ext = hi**K
machine, graphdef, params = create_machine(total_ansatz)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


$$\Psi(\mathbf{x}) = \det\begin{vmatrix}
\psi_1(\mathbf{x}_1) & \psi_2(\mathbf{x}_1) & \dots & \psi_K(\mathbf{x}_1) \\
\psi_1(\mathbf{x}_2) & \psi_2(\mathbf{x}_2) & \dots & \psi_K(\mathbf{x}_2) \\
\vdots & \vdots & \ddots & \vdots \\
\psi_1(\mathbf{x}_K) & \psi_2(\mathbf{x}_K) & \dots & \psi_K(\mathbf{x}_K)
\end{vmatrix}$$  
但是在代码上往往要使用对数域来避免数值溢出 因此:
$$\ln{\Psi(\mathbf{x})} = \ln\left[\, \det
\begin{pmatrix}
\psi_1(\mathbf{x}_1) & \dots & \psi_K(\mathbf{x}_1) \\
\vdots & \ddots & \vdots \\
\psi_1(\mathbf{x}_K) & \dots & \psi_K(\mathbf{x}_K)
\end{pmatrix}
\,\right]$$ 


In [4]:
tensor_edges = [(0,1),(2,3),(4,5),(6,7)]
total_ansatz = NESTotalAnsatz(n_spin_orbitals=4,n_states=2,hidden_dim=8,rngs=nnx.Rngs(13))
total_machine, total_graphdef, total_params = create_machine(total_ansatz)

In [5]:
sampler_state = init_sampler_state(hi_ext, 16,seed=42)
samples,sampler_state = mcmc_sampler_multichain(
    n_samples_per_chain=100,
    n_warmup=20,
    sampler_state=sampler_state,
    edges=tuple(tensor_edges),
    machine=machine,
    params=params,
)
samples.shape

TypeError: cannot reshape array of shape (1, 8) (size 8) into shape (3, 4) (size 12)

In [ ]:
def NES_loss_energy(ha, graphdef, params, x):
    total_model = nnx.merge(graphdef, params)
    log_psi_det, log_M = total_model(x)
    Psi_Matrix = jnp.exp(log_M)
    # 添加正则化项，防止矩阵奇异
    #Psi_Matrix += 1e-6 * jnp.eye(Psi_Matrix.shape[0])
    H_psi_x = Ham_Psi(ha, total_model, x)
    Psi_Matrix_inv = jnp.linalg.solve(Psi_Matrix, H_psi_x)
    return jnp.real(jnp.trace(Psi_Matrix_inv)), Psi_Matrix_inv

In [ ]:
NES_loss_energy(ha,graphdef, params, x)

In [ ]:
compute_nes_qgt(machine,graphdef, params, samples, diag_shift=0.001)[0].shape  # 458,458

In [ ]:
@partial(jax.vmap, in_axes=(None, None, None, 0))
def compute_local_energy_matrix_batch(ha: nk.operator.DiscreteOperator,graphdef, params, x_batch):
    loss_val, E_L = NES_loss_energy(ha, graphdef, params, x_batch)
    return E_L


In [ ]:
compute_local_energy_matrix_batch(ha, graphdef, params, samples.reshape(-1,K,4))

In [ ]:
grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                graphdef=graphdef,
                                                params=params,
                                                x_batch=samples.reshape(-1,2,4))
grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
grad_flat.shape

In [ ]:
grad_flat

In [ ]:
from typing import Any
from NES_VMC import compute_qgt
import time
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 50
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 20
N_ITER =100

# ======================
# 初始化 ONCE
# ======================
rngs = nnx.Rngs(21)
model = NESTotalAnsatz(4,K,12,rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (自然梯度下降法)")

print("超参数")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[]
}
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")
sampler_state = init_sampler_state(hi_ext, N_CHAINS, seed=21)  # 每次迭代换种子避免初始状态固定
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples,sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,
        edges=tuple(tensor_edges),
        machine=machine,
        params=params,
    )
    #samples = samples.reshape(-1,2,4)

    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 graphdef=graphdef,
                                                 params=params,
                                                 x_batch=samples.reshape(-1,K,4))
    grad = jax.tree_map(lambda x: x*2, grad)
    #model_output = log(\Psi(X)) 
    # qgt_reg,qgt_unravel_fun = compute_nes_qgt(machine, graphdef, params, samples, diag_shift=0.001) 
    # grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
    
    # # # 自然梯度求解
    # natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad)
    # grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 10 == 0 or step == N_ITER - 1:
        total_model =  nnx.merge(graphdef,params)
        log_Psi,log_M  = total_model(samples.reshape(-1,2,4))
        eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        history['step'].append(step)
        history['E_Lmatrix'].append(E_L_mean)
        history['samples'].append(samples)
        history['loss'].append(loss_mean)
        #history['loss'].append(loss_mean)
        #history['natural_grad'].append(natural_grad)
        history['grad_flat'].append(grad_flat)
        history['log_Psi'].append(log_Psi)
        history['log_M'].append(log_M)
        
        # history['energy'].append(loss_mean)
        # history['energy_std'].append(jnp.std(loss_mean))
        # history['error'].append(jnp.abs(loss_mean - E_fcis[0]))
        history['params'].append(params)
        print(f"Step {step:3d} | Loss: {loss_mean}｜eig_vals: {eig_vals}|grad_flat: {grad_flat[:4]}|E_L_mean: {E_L_mean[0,:]}")



end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
# print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
# print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
# print(f"绝对误差：{final_error:.6f} Ha")
# print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)

In [ ]:
matrix = history['E_Lmatrix'][6]
eig_vals, eig_vecs = jnp.linalg.eigh(matrix)
eig_vals

In [ ]:
nes_vmc_gradient(ha,graphdef,history['params'][6],x_batch=history['samples'][6].reshape(-1,2,4))

In [ ]:
test_samples = history['samples'][6].reshape(-1,8)

In [ ]:
def summary_sampler(samples:jnp.array):
    samples = samples.reshape(-1,8)
    tuple_test_samples = [tuple(x.tolist()) for x in samples]
    tuple_test_samples = tuple(tuple_test_samples)
    #amples = tuple(samples.tolist())
    counter = Counter(tuple_test_samples)
    return counter

In [ ]:
summary_sampler(history['samples'][69])

In [ ]:
import jax.numpy as jnp
from collections import Counter
import matplotlib.pyplot as plt
import imageio
import numpy as np
from tqdm import tqdm  # 显示进度条（可选）
plt.rcParams['font.family'] = ['Hiragino Sans GB']
# ---------------------- 你自己的函数（完美版，直接用） ----------------------
def summary_sampler(samples: jnp.array):
    # 重塑成 (N,8)，每行一个8维向量
    samples = samples.reshape(-1, 8)
    # 把每一行转成 tuple（可哈希，Counter可用）
    tuple_samples = [tuple(x.tolist()) for x in samples]
    # 统计频次
    counter = Counter(tuple_samples)
    return counter

# ---------------------- 绘制单张直方图 ----------------------
def plot_histogram(counter, step, save_path="temp.png"):
    # 提取数据
    keys = [str(k) for k in counter.keys()]   # 序列转字符串
    values = list(counter.values())           # 频次
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(keys, values, color='#4287f5', edgecolor='black', alpha=0.8)
    
    # 标注数值
    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.title(f"Step {step} - 8维样本分布直方图", fontsize=16)
    plt.xlabel("8维样本 (0/1序列)", fontsize=12)
    plt.ylabel("出现频次", fontsize=12)
    plt.xticks(rotation=30, ha='right')  # 旋转防止重叠
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

# ---------------------- 生成 GIF：从 step 0 ~ 70 动态演化 ----------------------
def make_evolution_gif(history, total_steps=69, gif_name="distribution_evolution.gif"):
    frames = []
    
    print("正在生成每一帧直方图...")
    for step in tqdm(range(65)):  # 0 ~ 70
        # 取出当前步的 samples
        samples = history['samples'][step]
        
        # 统计分布
        counter = summary_sampler(samples)
        
        # 保存临时图片
        plot_histogram(counter, step, "temp_frame.png")
        
        # 读入图片作为帧
        frames.append(imageio.imread("temp_frame.png"))
    
    # 合成GIF
    print("正在合成GIF...")
    imageio.mimsave(gif_name, frames, duration=1.2, loop=0)
    print(f"✅ GIF 已保存：{gif_name}")

# ---------------------- 【直接运行】 ----------------------
if __name__ == "__main__":
    # 运行这句即可！
    make_evolution_gif(history, total_steps=71)